In [1]:
from web3 import Web3

ganache_url = "http://127.0.0.1:7545"

web3 = Web3(
    Web3.HTTPProvider(ganache_url)
)

if web3.is_connected():
    print("Connected to Ganache successfully!")
else:
    print("Connection failed!")

Connected to Ganache successfully!


In [3]:
import json

contract_address = "0x8e1c7A22833Fe4699B7fba08754eE315d87731f4"

abi = json.loads("""
[
	{
		"inputs": [],
		"stateMutability": "nonpayable",
		"type": "constructor"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "dataType",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "dataValue",
				"type": "string"
			}
		],
		"name": "DataStored",
		"type": "event"
	},
	{
		"inputs": [],
		"name": "MAX_ENTRIES",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"name": "dataRecords",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "dataType",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "dataValue",
				"type": "string"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "index",
				"type": "uint256"
			}
		],
		"name": "getRecord",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "",
				"type": "string"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "getTotalRecords",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "owner",
		"outputs": [
			{
				"internalType": "address",
				"name": "",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "_deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "_dataType",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "_dataValue",
				"type": "string"
			}
		],
		"name": "storeData",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	}
]
""")

contract = web3.eth.contract(
    address=contract_address,
    abi=abi
)

web3.eth.default_account = web3.eth.accounts[0]

print("Smart contract connected successfully!")

Smart contract connected successfully!


In [4]:
total_records = contract.functions.getTotalRecords().call()

print(
    f"Total IoT records stored: {total_records}"
)

Total IoT records stored: 100


In [5]:
import pandas as pd

data = []

for i in range(total_records):

    record = contract.functions.getRecord(i).call()

    data.append({
        "timestamp": record[0],
        "device_id": record[1],
        "data_type": record[2],
        "data_value": record[3]
    })

df = pd.DataFrame(data)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    unit="s"
)

print(df.head())

            timestamp device_id     data_type data_value
0 2026-05-24 05:18:25   TEST001   Temperature       22.5
1 2026-05-24 08:45:02   TEST002    Heart Rate     78 bpm
2 2026-05-24 09:03:43    PAT440  Oxygen Level        99%
3 2026-05-24 09:03:43    PAT434    Heart Rate     93 bpm
4 2026-05-24 09:03:43    PAT355    Heart Rate     79 bpm


In [6]:
import numpy as np

df["numeric_value"] = (
    df["data_value"]
    .str.extract(r'(\d+\.?\d*)')
    .astype(float)
)

df.fillna(0, inplace=True)

print(df.head())

            timestamp device_id     data_type data_value  numeric_value
0 2026-05-24 05:18:25   TEST001   Temperature       22.5           22.5
1 2026-05-24 08:45:02   TEST002    Heart Rate     78 bpm           78.0
2 2026-05-24 09:03:43    PAT440  Oxygen Level        99%           99.0
3 2026-05-24 09:03:43    PAT434    Heart Rate     93 bpm           93.0
4 2026-05-24 09:03:43    PAT355    Heart Rate     79 bpm           79.0


In [7]:
df.to_csv(
    "cleaned_iot_data.csv",
    index=False
)

print(
    "Cleaned IoT data saved successfully!"
)

Cleaned IoT data saved successfully!
